<a href="https://colab.research.google.com/github/DavinciDreams/JuliaGPT/blob/main/juliadistill.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JuliaFlux Distill — Small Model Training

Train a small LLaMA-style language model on a combined humanities + wikitext corpus.

**Architecture:** RoPE, SwiGLU, GQA, RMSNorm, weight tying  
**Data:** Combined humanities + wikitext (~3M lines, 549MB)  
**Tokenizer:** BPE with 4000 vocab  
**Framework:** Julia + Flux.jl

---
## 1. Setup

Set your API tokens in a `.env` file or directly in the cell below.  
Required: `HF_TOKEN` (for downloading data), optional: `WANDB_API_KEY`, `HF_REPO`.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Option 1: Set tokens directly (uncomment and fill in)
# ══════════════════════════════════════════════════════════════════
# ENV["HF_TOKEN"]       = "hf_..."
# ENV["WANDB_API_KEY"]  = "..."
# ENV["HF_REPO"]        = "LisaMegaWatts/juliadistill-v2"
# ENV["HF_DATA_REPO"]   = "LisaMegaWatts/philosophy-corpus"

# ══════════════════════════════════════════════════════════════════
# Option 2: Load from .env file (recommended)
# ══════════════════════════════════════════════════════════════════
function load_dotenv(path=".env")
    isfile(path) || return 0
    n = 0
    for line in eachline(path)
        line = strip(line)
        (isempty(line) || startswith(line, '#')) && continue
        m = match(r"^([A-Za-z_][A-Za-z0-9_]*)=(.*)", line)
        m === nothing && continue
        key = m.captures[1]
        val = strip(m.captures[2], ['"', '\'', ' '])
        ENV[key] = val
        n += 1
    end
    return n
end

n = load_dotenv()
n > 0 && println("Loaded $n vars from .env")

# ── Validate ──
HF_TOKEN      = get(ENV, "HF_TOKEN", "")
HF_REPO_ID    = get(ENV, "HF_REPO", "LisaMegaWatts/juliadistill-v2")
HF_DATA_REPO  = get(ENV, "HF_DATA_REPO", "LisaMegaWatts/philosophy-corpus")

println("HF Token:  ", isempty(HF_TOKEN) ? "NOT SET" : "$(HF_TOKEN[1:min(8,end)])...")
println("Model Repo: ", HF_REPO_ID)
println("Data Repo:  ", HF_DATA_REPO)

---
## 2. Packages

In [ ]:
import Pkg
Pkg.add(["Flux", "Zygote", "Optimisers", "CUDA", "cuDNN",
         "NNlib", "JLD2", "JSON3", "HTTP", "SHA",
         "Downloads", "HuggingFaceApi"], io=devnull)
Pkg.instantiate()

using Flux, Zygote, Optimisers, CUDA, NNlib, Random, JLD2, JSON3, Printf
using Flux: softmax
using LinearAlgebra: triu
using Statistics: mean
using Random: MersenneTwister, shuffle!
using Downloads, HTTP

println("Packages loaded ✓")

In [ ]:
# ── GPU Detection ──
if CUDA.functional()
    device = gpu
    VRAM_GB = CUDA.total_memory() / 1024^3
    println("GPU: $(CUDA.name(CUDA.device())), VRAM=$(round(VRAM_GB, digits=1)) GB")
else
    device = cpu
    VRAM_GB = 0.0
    println("CPU mode (no GPU detected)")
end

---
## 3. HuggingFace + W&B Helpers

In [ ]:
# ══════════════════════════════════════════════════════════════════
# HuggingFace Hub helpers — pure Julia (no Python, no pip)
# Uses HuggingFaceApi.jl for dataset downloads, HTTP.jl for uploads
# ══════════════════════════════════════════════════════════════════

# ── Login note ──
# HuggingFaceApi.login() is interactive (username/password), NOT token-based.
# Pass auth_token=HF_TOKEN directly to hf_hub_download() instead.
if !isempty(HF_TOKEN)
    println("HuggingFace: token set (will pass auth_token to API calls)")
else
    println("HuggingFace: no token set (public repos only)")
end

import Base64
using SHA

# ── Download files from HuggingFace ──
# HuggingFaceApi.hf_hub_download works for repo_type="dataset" but has a bug
# with repo_type="model" (TypeError). HTTP fallback handles both.
function hf_download(repo_id::String, filename::String;
                     local_dir::String=".", repo_type::String="dataset")
    local_path = joinpath(local_dir, filename)
    isfile(local_path) && return local_path
    mkpath(local_dir)
    try
        path = HuggingFaceApi.hf_hub_download(repo_id, filename;
                    repo_type=repo_type, auth_token=HF_TOKEN)
        cp(path, local_path; force=true)
        println("  Downloaded: $filename ($(filesize(local_path)) bytes)")
    catch e
        # Fallback: direct HTTP download (works for both dataset and model repos)
        prefix = repo_type == "dataset" ? "datasets/" : ""
        url = "https://huggingface.co/$(prefix)$(repo_id)/resolve/main/$(filename)"
        headers = isempty(HF_TOKEN) ? Pair{String,String}[] : ["Authorization" => "Bearer $HF_TOKEN"]
        Downloads.download(url, local_path; headers)
        println("  Downloaded (HTTP fallback): $filename ($(filesize(local_path)) bytes)")
    end
    return local_path
end

# ── Upload files to HuggingFace ──
# Small files (<5MB): JSON commit API with base64 content
# Large files (>=5MB): Git LFS batch API (3-step: batch → upload blob → commit pointer)
function hf_upload(repo_id::String, local_path::String;
                   remote_path::String="", repo_type::String="model")
    isempty(HF_TOKEN) && (@warn "Cannot upload: no HF_TOKEN set"; return)
    rp = isempty(remote_path) ? basename(local_path) : remote_path
    prefix = repo_type == "model" ? "models" : "datasets"
    base_url = "https://huggingface.co/api/$(prefix)/$(repo_id)"
    data = read(local_path)
    auth_headers = ["Authorization" => "Bearer $HF_TOKEN"]

    if length(data) < 5_000_000
        # Small files: base64 JSON commit
        encoded = Base64.base64encode(data)
        headers = vcat(auth_headers, ["Content-Type" => "application/json"])
        body = JSON3.write(Dict(
            "summary" => "Upload $rp",
            "files" => [Dict("path" => rp, "content" => encoded, "encoding" => "base64")]
        ))
        try
            HTTP.post("$(base_url)/commit/main", headers, body)
            println("Pushed $local_path -> $repo_id/$rp ($(length(data)) bytes)")
        catch e
            @warn "Upload failed: $e"
        end
    else
        # Large files: Git LFS batch upload
        file_sha = bytes2hex(sha256(data))
        headers = vcat(auth_headers, ["Content-Type" => "application/json"])
        println("Uploading $(round(length(data)/1024/1024, digits=1)) MB via LFS...")

        # Step 1: LFS batch request to get upload URL
        lfs_url = "https://huggingface.co/$(repo_id).git/info/lfs/objects/batch"
        lfs_body = JSON3.write(Dict(
            "operation" => "upload",
            "transfers" => ["basic"],
            "objects" => [Dict("oid" => file_sha, "size" => length(data))]
        ))
        lfs_headers = vcat(auth_headers, [
            "Content-Type" => "application/vnd.git-lfs+json",
            "Accept" => "application/vnd.git-lfs+json"
        ])
        try
            resp = HTTP.post(lfs_url, lfs_headers, lfs_body)
            lfs_resp = JSON3.read(String(resp.body))
            obj = lfs_resp.objects[1]

            # Step 2: Upload blob to LFS storage
            if haskey(obj, :actions) && haskey(obj.actions, :upload)
                upload_action = obj.actions.upload
                upload_hdrs = Pair{String,String}[]
                if haskey(upload_action, :header)
                    for (k, v) in pairs(upload_action.header)
                        push!(upload_hdrs, string(k) => string(v))
                    end
                end
                HTTP.put(string(upload_action.href), upload_hdrs, data)
            end

            # Step 3: Commit LFS pointer
            commit_body = JSON3.write(Dict(
                "summary" => "Upload $rp ($(round(length(data)/1024/1024, digits=1)) MB)",
                "lfsFiles" => [Dict("path" => rp, "algo" => "sha256", "oid" => file_sha, "size" => length(data))]
            ))
            HTTP.post("$(base_url)/commit/main", headers, commit_body)
            println("Pushed $local_path -> $repo_id/$rp ($(length(data)) bytes, LFS)")
        catch e
            @warn "LFS upload failed: $e"
        end
    end
end

function hf_push(repo_id::String, local_path::String; remote_path::String="")
    hf_upload(repo_id, local_path; remote_path)
end

function hf_pull(repo_id::String, filename::String; local_dir::String=".")
    hf_download(repo_id, filename; local_dir=local_dir, repo_type="model")
end

function hf_push_checkpoint(repo_id::String; checkpoint_path::String="checkpoints/best_model.jld2")
    isfile(checkpoint_path) || error("Checkpoint not found: $checkpoint_path")
    hf_push(repo_id, checkpoint_path)
end

function hf_create_repo(repo_id::String)
    isempty(HF_TOKEN) && return
    url = "https://huggingface.co/api/repos/create"
    headers = [
        "Authorization" => "Bearer $HF_TOKEN",
        "Content-Type" => "application/json",
    ]
    body = JSON3.write(Dict("name" => split(repo_id, "/")[end], "type" => "model", "private" => false))
    try
        HTTP.post(url, headers, body)
        println("Created HF repo: $repo_id")
    catch
        println("HF repo already exists or creation skipped: $repo_id")
    end
end

function hf_sync(local_path::String)
    isempty(HF_REPO_ID) && return
    try
        hf_push(HF_REPO_ID, local_path)
    catch e
        println("  HF sync failed: $e")
    end
end

# ═══════════════════════════════════════════════════════════════
# W&B logging via HTTP API (pure Julia — no Python, no pip)
# Uses W&B's internal API: GraphQL for run creation, file_stream for metrics
# ═══════════════════════════════════════════════════════════════

mutable struct WandbRun
    entity::String
    project::String
    run_id::String
    run_name::String
    api_key::String
    base_url::String
    step::Int
    active::Bool
    config::Dict{String,Any}
    log_buffer::Vector{Dict{String,Any}}
    flush_interval::Int  # flush every N log calls
end

const _WANDB_RUN = Ref{Union{WandbRun, Nothing}}(nothing)

function _wandb_graphql(api_key::String, query::String, variables::Dict; base_url::String="https://api.wandb.ai")
    auth = Base64.base64encode("api:$api_key")
    headers = [
        "Authorization" => "Basic $auth",
        "Content-Type" => "application/json"
    ]
    body = JSON3.write(Dict("query" => query, "variables" => variables))
    resp = HTTP.post("$base_url/graphql", headers, body; status_exception=false)
    if resp.status != 200
        @warn "W&B GraphQL request failed ($(resp.status))"
        return nothing
    end
    return JSON3.read(String(resp.body))
end

function _wandb_get_entity(api_key::String; base_url::String="https://api.wandb.ai")
    result = _wandb_graphql(api_key, "query { viewer { entity } }", Dict(); base_url=base_url)
    result === nothing && return ""
    try
        return string(result.data.viewer.entity)
    catch
        return ""
    end
end

function _wandb_create_run(run::WandbRun)
    variables = Dict(
        "entity" => run.entity,
        "modelName" => run.project,
        "id" => run.run_id,
        "displayName" => run.run_name,
        "config" => JSON3.write(Dict(k => Dict("value" => v) for (k, v) in run.config))
    )
    query = """
    mutation UpsertBucket(\$entity: String, \$modelName: String, \$id: String, \$displayName: String, \$config: JSONString) {
        upsertBucket(input: {entityName: \$entity, modelName: \$modelName, name: \$id, displayName: \$displayName, config: \$config}) {
            bucket { id name displayName project { id name } }
        }
    }
    """
    result = _wandb_graphql(run.api_key, query, variables; base_url=run.base_url)
    result === nothing && return false
    try
        if hasproperty(result, :errors) && !isempty(result.errors)
            @warn "W&B run creation error: $(result.errors[1].message)"
            return false
        end
    catch; end
    return true
end

function _wandb_flush(run::WandbRun)
    isempty(run.log_buffer) && return
    auth = Base64.base64encode("api:$(run.api_key)")
    headers = [
        "Authorization" => "Basic $auth",
        "Content-Type" => "application/json"
    ]
    # Build history rows from buffer
    rows = [JSON3.write(entry) for entry in run.log_buffer]
    offset = run.step - length(run.log_buffer)
    body = JSON3.write(Dict(
        "files" => Dict(
            "wandb-history.jsonl" => Dict(
                "offset" => max(0, offset),
                "content" => rows
            )
        )
    ))
    url = "$(run.base_url)/files/$(run.entity)/$(run.project)/$(run.run_id)/file_stream"
    try
        HTTP.post(url, headers, body; status_exception=false)
    catch e
        @warn "W&B flush failed: $e"
    end
    empty!(run.log_buffer)
end

function wandb_init(; project::String="JuliaDistillGPT", name::String="", config::Dict=Dict{String,Any}())
    api_key = get(ENV, "WANDB_API_KEY", "")
    if isempty(api_key)
        println("W&B: no API key found, logging disabled")
        return
    end
    base_url = get(ENV, "WANDB_BASE_URL", "https://api.wandb.ai")

    # Get entity (username)
    entity = _wandb_get_entity(api_key; base_url=base_url)
    if isempty(entity)
        @warn "W&B: could not resolve entity, logging disabled"
        return
    end

    # Generate run ID and name
    run_id = lowercase(randstring(8))
    run_name = isempty(name) ? "julia-$(run_id)" : name

    run = WandbRun(entity, project, run_id, run_name, api_key, base_url,
                   0, true, Dict{String,Any}(string(k) => v for (k,v) in config),
                   Dict{String,Any}[], 10)

    if _wandb_create_run(run)
        _WANDB_RUN[] = run
        println("W&B: run started → https://wandb.ai/$entity/$project/runs/$run_id")
    else
        println("W&B: run creation failed, logging disabled")
    end
end

function wandb_log(; kwargs...)
    run = _WANDB_RUN[]
    run === nothing && return
    !run.active && return

    entry = Dict{String,Any}(string(k) => v for (k, v) in kwargs)
    if !haskey(entry, "_step")
        run.step += 1
        entry["_step"] = run.step
    end
    push!(run.log_buffer, entry)

    if length(run.log_buffer) >= run.flush_interval
        _wandb_flush(run)
    end
end

function wandb_finish()
    run = _WANDB_RUN[]
    run === nothing && return
    _wandb_flush(run)
    run.active = false
    _WANDB_RUN[] = nothing
    println("W&B: run $(run.run_name) finished → https://wandb.ai/$(run.entity)/$(run.project)/runs/$(run.run_id)")
end

println("HuggingFace + W&B helpers defined (pure Julia, LFS-capable)")

---
## 4. Hyperparameters

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Auto-select model size based on available VRAM
# ══════════════════════════════════════════════════════════════════

if VRAM_GB >= 24.0
    COMPUTE_TIER = :large
    block_size     = 256
    n_embd         = 256
    n_head         = 4       # HEAD_DIM = 64
    n_kv_head      = 2
    n_layer        = 8
    batch_size     = 64
    dropout        = 0.1
elseif VRAM_GB >= 12.0
    # ── T4 tier (16GB) or RTX 3060 (12GB) ──
    COMPUTE_TIER = :t4
    block_size     = 256
    n_embd         = 256
    n_head         = 4       # HEAD_DIM = 64
    n_kv_head      = 2
    n_layer        = 6
    batch_size     = 32
    dropout        = 0.1
elseif VRAM_GB >= 8.0
    COMPUTE_TIER = :medium
    block_size     = 256
    n_embd         = 256
    n_head         = 4       # HEAD_DIM = 64
    n_kv_head      = 2
    n_layer        = 4
    batch_size     = 16
    dropout        = 0.1
else
    COMPUTE_TIER = :small
    block_size     = 128
    n_embd         = 128
    n_head         = 4       # HEAD_DIM = 32
    n_kv_head      = 2
    n_layer        = 4
    batch_size     = 16
    dropout        = 0.1
end

# ── Fixed architecture params ──
bias           = false
rope_base      = 10000.0f0

# ── Training ──
learning_rate  = 3e-4
target_epochs  = 3
eval_iters     = COMPUTE_TIER == :small ? 100 : COMPUTE_TIER == :medium ? 80 : 50
warmup_frac    = 0.05
min_lr         = 1e-5
max_grad_norm  = 1.0f0

# ── KD params (stored in checkpoints for compatibility) ──
kd_alpha       = 0.5f0
kd_temperature = 4.0f0
teacher_logits_available = false

println("═══ Compute Tier: $COMPUTE_TIER (VRAM=$(round(VRAM_GB, digits=1)) GB) ═══")
println("Model: n_embd=$n_embd, n_layer=$n_layer, n_head=$n_head (Q), n_kv_head=$n_kv_head (KV)")
println("  head_dim=$(n_embd ÷ n_head), GQA ratio=$(n_head ÷ n_kv_head):1, block_size=$block_size")
println("Training: batch=$batch_size, lr=$learning_rate, target_epochs=$target_epochs")

---
## 5. Load Data

In [ ]:
# ── Load combined humanities + wikitext corpus ──
DATA_DIR = "data"
train_file = joinpath(DATA_DIR, "train.txt")
val_file   = joinpath(DATA_DIR, "val.txt")

if !isfile(train_file) || !isfile(val_file)
    println("Data not found locally, downloading from $HF_DATA_REPO ...")
    mkpath(DATA_DIR)
    try
        hf_download(HF_DATA_REPO, "train.txt"; local_dir=DATA_DIR, repo_type="dataset")
        hf_download(HF_DATA_REPO, "val.txt"; local_dir=DATA_DIR, repo_type="dataset")
    catch e
        @warn "Download failed: $e"
    end
end

if !isfile(train_file) || !isfile(val_file)
    error("No data found in $DATA_DIR/! Push train.txt + val.txt to $HF_DATA_REPO")
end

train_text = read(train_file, String)
val_text   = read(val_file, String)

println("Data loaded from $DATA_DIR/")
println("  train: $(length(train_text)) chars ($(count('\n', train_text)) lines, $(round(length(train_text)/1e6, digits=1))MB)")
println("  val:   $(length(val_text)) chars ($(count('\n', val_text)) lines, $(round(length(val_text)/1e6, digits=1))MB)")

# ── Download tokenizer.json if missing ──
tokenizer_file = joinpath(DATA_DIR, "tokenizer.json")
if !isfile(tokenizer_file)
    try
        hf_download(HF_DATA_REPO, "tokenizer.json"; local_dir=DATA_DIR, repo_type="dataset")
    catch
        println("  No tokenizer.json available (will use char-level tokenizer)")
    end
end

---
## 6. Tokenizer

In [ ]:
# ── Tokenizer: BPE with character-level fallback ──
using JSON3

TOKENIZER_PATH = joinpath(DATA_DIR, "tokenizer.json")
USE_BPE = isfile(TOKENIZER_PATH)

if USE_BPE
    println("Loading BPE tokenizer from $TOKENIZER_PATH ...")
    tok_raw = read(TOKENIZER_PATH, String)
    tok_json = JSON3.read(tok_raw)

    bpe_vocab = Dict{String, Int}()
    for (tok_str, id) in pairs(tok_json.model.vocab)
        bpe_vocab[string(tok_str)] = Int(id) + 1
    end

    bpe_merges = Vector{Tuple{String,String}}()
    for merge_entry in tok_json.model.merges
        if merge_entry isa AbstractVector && length(merge_entry) >= 2
            # New format: ["token_a", "token_b"]
            push!(bpe_merges, (String(merge_entry[1]), String(merge_entry[2])))
        else
            # Old format: "token_a token_b"
            parts = split(string(merge_entry), " ", limit=2)
            if length(parts) == 2
                push!(bpe_merges, (String(parts[1]), String(parts[2])))
            end
        end
    end

    bpe_id_to_token = Dict{Int, String}(id => tok for (tok, id) in bpe_vocab)
    global vocab_size = length(bpe_vocab)

    # ── GPT-2 byte-to-unicode mapping ──
    function build_byte_to_unicode()
        bs = UInt8[]
        cs = Char[]
        for r in [0x21:0x7e, 0xa1:0xac, 0xae:0xff]
            for b in r
                push!(bs, b)
                push!(cs, Char(b))
            end
        end
        n = 0
        for b in 0x00:0xff
            if b ∉ bs
                push!(bs, b)
                push!(cs, Char(256 + n))
                n += 1
            end
        end
        return Dict(bs[i] => string(cs[i]) for i in eachindex(bs))
    end

    const _b2u = build_byte_to_unicode()
    const _u2b = Dict(v[1] => k for (k, v) in _b2u)

    # ── Merge rank lookup: O(1) pair check instead of scanning all merges ──
    const _merge_rank = Dict{Tuple{String,String}, Int}(
        (a, b) => i for (i, (a, b)) in enumerate(bpe_merges)
    )

    function bpe_encode_word(word::Vector{String})
        tokens = copy(word)
        while length(tokens) >= 2
            best_rank = typemax(Int)
            best_pair = ("", "")
            for i in 1:length(tokens)-1
                rank = get(_merge_rank, (tokens[i], tokens[i+1]), typemax(Int))
                if rank < best_rank
                    best_rank = rank
                    best_pair = (tokens[i], tokens[i+1])
                end
            end
            best_rank == typemax(Int) && break
            a, b = best_pair
            new_tokens = String[]
            i = 1
            while i <= length(tokens)
                if i < length(tokens) && tokens[i] == a && tokens[i+1] == b
                    push!(new_tokens, a * b)
                    i += 2
                else
                    push!(new_tokens, tokens[i])
                    i += 1
                end
            end
            tokens = new_tokens
        end
        return tokens
    end

    const _GPT2_PAT = r"'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"

    const _word_cache = Dict{String, Vector{Int}}()

    function encode(s::String)
        ids = Int[]
        for m in eachmatch(_GPT2_PAT, s)
            word = m.match
            cached = get(_word_cache, word, nothing)
            if cached !== nothing
                append!(ids, cached)
            else
                word_bytes = Vector{UInt8}(word)
                chars = [_b2u[b] for b in word_bytes]
                tokens = bpe_encode_word(chars)
                word_ids = Int[]
                for tok in tokens
                    id = get(bpe_vocab, tok, nothing)
                    id !== nothing && push!(word_ids, id)
                end
                _word_cache[word] = word_ids
                append!(ids, word_ids)
            end
        end
        return ids
    end

    function decode(ids::Vector{Int})
        text = join(get(bpe_id_to_token, id, "") for id in ids)
        bytes = UInt8[_u2b[c] for c in text if haskey(_u2b, c)]
        return String(bytes)
    end

    println("BPE tokenizer: vocab_size=$vocab_size, $(length(bpe_merges)) merges")

else
    println("No tokenizer.json found — using character-level tokenizer")

    full_text = train_text * "\n" * val_text
    chars = sort(unique(full_text))
    filter!(c -> c != '\n', chars)
    global vocab_size = length(chars)

    stoi = Dict(c => i for (i, c) in enumerate(chars))
    itos = Dict(i => c for (i, c) in enumerate(chars))

    encode(s::String) = [stoi[c] for c in s if haskey(stoi, c)]
    decode(ids::Vector{Int}) = join(itos[i] for i in ids)

    println("Char-level tokenizer: vocab_size=$vocab_size -> [$(join(chars))]")
end

println("Encoding training data...")
flush(stdout)
t_enc = time()
train_clean = replace(strip(train_text), '\n' => ' ')
val_clean   = replace(strip(val_text), '\n' => ' ')

global train_data = encode(train_clean)
global val_data   = encode(val_clean)
enc_time = round(time() - t_enc, digits=1)
println("Encoding done in $(enc_time)s ($(USE_BPE ? "$(length(_word_cache)) unique words cached" : "char-level"))")

println("\nTrain: $(length(train_data)) tokens")
println("Val:   $(length(val_data)) tokens")
println("Total: $(length(train_data) + length(val_data)) tokens")

# ── Compute training schedule from data size ──
tokens_per_step = batch_size * block_size
steps_per_epoch = length(train_data) ÷ tokens_per_step
global max_iters = max(500, target_epochs * steps_per_epoch)
global eval_interval = max(50, max_iters ÷ 20)
global warmup_iters = max(50, round(Int, warmup_frac * max_iters))

println("\n── Training schedule (computed from data) ──")
println("  tokens_per_step: $tokens_per_step (batch=$batch_size × block=$block_size)")
println("  steps_per_epoch: $steps_per_epoch")
println("  target_epochs:   $target_epochs")
println("  max_iters:       $max_iters")
println("  eval_interval:   $eval_interval")
println("  warmup_iters:    $warmup_iters (LR warmup)")

---
## 7. DataLoader

In [ ]:
# ── Shuffled Sequential DataLoader ──
# Guarantees full coverage of the dataset each epoch instead of random sampling
# which can overtrain some passages while never seeing others.

mutable struct ShuffledDataLoader
    data::Vector{Int}
    block_size::Int
    batch_size::Int
    positions::Vector{Int}      # shuffled valid start indices
    pos_idx::Int                # current index into positions
    epoch::Int                  # current epoch number
end

function ShuffledDataLoader(data::Vector{Int}, block_size::Int, batch_size::Int)
    positions = collect(1:(length(data) - block_size))
    rng = MersenneTwister(1337 + 1)
    shuffle!(rng, positions)
    ShuffledDataLoader(data, block_size, batch_size, positions, 1, 1)
end

function next_batch!(loader::ShuffledDataLoader)
    bs = loader.batch_size
    # Start new epoch if not enough positions left for a full batch
    if loader.pos_idx + bs - 1 > length(loader.positions)
        loader.epoch += 1
        rng = MersenneTwister(1337 + loader.epoch)
        shuffle!(rng, loader.positions)
        loader.pos_idx = 1
    end

    indices = loader.positions[loader.pos_idx:loader.pos_idx + bs - 1]
    loader.pos_idx += bs

    d = loader.data
    x = hcat([d[i:i+loader.block_size-1] for i in indices]...)
    y = hcat([d[i+1:i+loader.block_size] for i in indices]...)
    x = permutedims(x)
    y = permutedims(y)
    return x |> device, y |> device
end

# ── Create loader ──
global train_loader = ShuffledDataLoader(train_data, block_size, batch_size)

n_positions = length(train_loader.positions)
println("ShuffledDataLoader: $(n_positions) positions, ~$(n_positions ÷ batch_size) steps/epoch")

function get_batch(split="train")
    if split == "val"
        # Val stays random (standard practice)
        d = val_data
        ix = rand(1:length(d) - block_size, batch_size)
        x = hcat([d[i:i+block_size-1] for i in ix]...)
        y = hcat([d[i+1:i+block_size] for i in ix]...)
        x = permutedims(x)
        y = permutedims(y)
        return x |> device, y |> device
    end

    return next_batch!(train_loader)
end

---
## 8. Model Architecture

LLaMA-style transformer: RoPE, SwiGLU FFN, Grouped Query Attention, RMSNorm, weight-tied embedding.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# ALL MODEL STRUCTS IN ONE CELL (Julia structs cannot be redefined)
# Same LLaMA-style architecture as juliaflux_v2.ipynb
# ══════════════════════════════════════════════════════════════════

using NNlib: batched_mul

const CAUSAL_MASK = triu(fill(typemin(Float32), block_size, block_size), 1)
const CAUSAL_MASK_GPU = CUDA.functional() ? cu(CAUSAL_MASK) : CAUSAL_MASK

const HEAD_DIM = n_embd ÷ n_head

function precompute_rope_freqs(head_dim::Int, max_seq_len::Int; base::Float32 = 10000.0f0)
    half_dim = head_dim ÷ 2
    freqs = Float32[1.0f0 / (base ^ (Float32(2 * (i - 1)) / Float32(head_dim))) for i in 1:half_dim]
    positions = Float32.(collect(0:max_seq_len-1))
    angles = freqs * positions'
    return cos.(angles), sin.(angles)
end

const ROPE_COS, ROPE_SIN = precompute_rope_freqs(HEAD_DIM, block_size; base=rope_base)
const ROPE_COS_GPU = CUDA.functional() ? cu(ROPE_COS) : ROPE_COS
const ROPE_SIN_GPU = CUDA.functional() ? cu(ROPE_SIN) : ROPE_SIN

function apply_rope(x, cos_f, sin_f, T::Int)
    d = size(x, 1) ÷ 2
    x1 = x[1:d, :, :]
    x2 = x[d+1:2d, :, :]
    c = cos_f[:, 1:T]
    s = sin_f[:, 1:T]
    return vcat(x1 .* c .- x2 .* s, x1 .* s .+ x2 .* c)
end

struct RMSNorm{W <: AbstractVector}
    weight::W
    eps::Float32
end

Flux.@layer RMSNorm

function RMSNorm(dim::Int; eps::Float32 = 1.0f-6)
    RMSNorm(ones(Float32, dim), eps)
end

function (rn::RMSNorm)(x)
    rms = sqrt.(mean(x .^ 2, dims=1) .+ rn.eps)
    return (x ./ rms) .* rn.weight
end

struct SwiGLUFFN
    w_gate::Dense
    w_up::Dense
    w_down::Dense
    drop::Dropout
end

Flux.@layer SwiGLUFFN

function SwiGLUFFN(n_embd::Int; bias=false, dropout=0.0)
    raw_inner = Int(floor(4 * n_embd * 2 / 3))
    inner_dim = max(64, 64 * div(raw_inner + 32, 64))
    SwiGLUFFN(
        Dense(n_embd => inner_dim; bias),
        Dense(n_embd => inner_dim; bias),
        Dense(inner_dim => n_embd; bias),
        Dropout(dropout)
    )
end

function (ff::SwiGLUFFN)(x)
    ff.drop(ff.w_down(NNlib.swish(ff.w_gate(x)) .* ff.w_up(x)))
end

struct CausalSelfAttention
    wq::Dense
    wkv::Dense
    proj::Dense
    n_head::Int
    n_kv_head::Int
end

Flux.@layer CausalSelfAttention trainable=(wq, wkv, proj)

function CausalSelfAttention(n_embd::Int, n_head::Int, n_kv_head::Int; bias=false)
    head_dim = n_embd ÷ n_head
    kv_dim = head_dim * n_kv_head
    CausalSelfAttention(
        Dense(n_embd => n_embd; bias),
        Dense(n_embd => 2 * kv_dim; bias),
        Dense(n_embd => n_embd; bias),
        n_head,
        n_kv_head
    )
end

function (attn::CausalSelfAttention)(x)
    C, T, B = size(x)
    nh = attn.n_head
    nkv = attn.n_kv_head
    hs = C ÷ nh
    kv_dim = hs * nkv
    groups = nh ÷ nkv

    q = attn.wq(x)
    kv = attn.wkv(x)
    k = kv[1:kv_dim, :, :]
    v = kv[kv_dim+1:2*kv_dim, :, :]

    q = reshape(permutedims(reshape(q, hs, nh, T, B), (1, 3, 2, 4)), hs, T, nh * B)
    k = reshape(permutedims(reshape(k, hs, nkv, T, B), (1, 3, 2, 4)), hs, T, nkv * B)
    v = reshape(permutedims(reshape(v, hs, nkv, T, B), (1, 3, 2, 4)), hs, T, nkv * B)

    cos_f = x isa CuArray ? ROPE_COS_GPU : ROPE_COS
    sin_f = x isa CuArray ? ROPE_SIN_GPU : ROPE_SIN
    q = apply_rope(q, cos_f, sin_f, T)
    k = apply_rope(k, cos_f, sin_f, T)

    if groups > 1
        k_4d = reshape(k, hs, T, nkv, B)
        v_4d = reshape(v, hs, T, nkv, B)
        k_rep = repeat(reshape(k_4d, hs, T, nkv, 1, B), 1, 1, 1, groups, 1)
        v_rep = repeat(reshape(v_4d, hs, T, nkv, 1, B), 1, 1, 1, groups, 1)
        k = reshape(permutedims(k_rep, (1, 2, 4, 3, 5)), hs, T, nh * B)
        v = reshape(permutedims(v_rep, (1, 2, 4, 3, 5)), hs, T, nh * B)
    end

    scale = Float32(1 / sqrt(hs))
    wei = batched_mul(permutedims(q, (2, 1, 3)), k) .* scale

    mask = x isa CuArray ? CAUSAL_MASK_GPU[1:T, 1:T] : CAUSAL_MASK[1:T, 1:T]
    wei = wei .+ mask
    wei = softmax(wei; dims=2)

    out = batched_mul(v, permutedims(wei, (2, 1, 3)))
    out = reshape(permutedims(reshape(out, hs, T, nh, B), (1, 3, 2, 4)), C, T, B)

    attn.proj(out)
end

struct TransformerBlock
    ln1::RMSNorm
    attn::CausalSelfAttention
    ln2::RMSNorm
    ffwd::SwiGLUFFN
end

Flux.@layer TransformerBlock

function TransformerBlock(n_embd::Int, n_head::Int, n_kv_head::Int; dropout=0.0)
    TransformerBlock(
        RMSNorm(n_embd),
        CausalSelfAttention(n_embd, n_head, n_kv_head),
        RMSNorm(n_embd),
        SwiGLUFFN(n_embd; dropout)
    )
end

function (block::TransformerBlock)(x)
    x = x .+ block.attn(block.ln1(x))
    x = x .+ block.ffwd(block.ln2(x))
    x
end

struct TiedDense{W <: AbstractMatrix}
    weight_ref::W
end

Flux.@layer TiedDense trainable=()

function (td::TiedDense)(x)
    C, T, B = size(x)
    W = td.weight_ref
    x_flat = reshape(x, C, T * B)
    out = W' * x_flat
    reshape(out, size(W, 2), T, B)
end

struct GPT
    wte::Embedding
    drop::Dropout
    blocks::Chain
    ln_f::RMSNorm
    lm_head::TiedDense
end

Flux.@layer GPT

function GPT(; vocab_size, n_embd, block_size, n_layer, n_head, n_kv_head, dropout=0.1)
    wte = Embedding(vocab_size => n_embd)
    GPT(
        wte,
        Dropout(dropout),
        Chain([TransformerBlock(n_embd, n_head, n_kv_head; dropout) for _ in 1:n_layer]...),
        RMSNorm(n_embd),
        TiedDense(wte.weight)
    )
end

function (m::GPT)(idx)
    B, T = size(idx)
    tok = permutedims(m.wte(idx), (1, 3, 2))
    x = m.drop(tok)
    x = m.blocks(x)
    x = m.ln_f(x)
    m.lm_head(x)
end

println("Student model structs defined (same architecture as juliaflux_v2)")
println("SOTA: RoPE, SwiGLU, GQA ($(n_head)Q/$(n_kv_head)KV), RMSNorm, weight tying")

In [ ]:
model = GPT(;
    vocab_size = vocab_size,
    n_embd     = n_embd,
    block_size = block_size,
    n_layer    = n_layer,
    n_head     = n_head,
    n_kv_head  = n_kv_head,
    dropout    = dropout
) |> device

n_params = sum(length, Flux.trainables(model))
println("Model created on $device")
println("Parameters: $(n_params) ($(round(n_params/1e6, digits=2))M)")

if CUDA.functional()
    println("GPU memory: $(round(CUDA.used_memory() / 1024^2, digits=1)) MB")
end

# Smoke test
x_test, y_test = get_batch("train")
logits_test = model(x_test)
println("Forward pass OK — logits: $(size(logits_test))")
@assert size(logits_test, 1) == vocab_size

---
## 9. Loss Function

In [ ]:
# ── Standard cross-entropy loss ──
function standard_loss(student_logits, hard_targets)
    Flux.logitcrossentropy(student_logits, hard_targets)
end
println("Loss function defined ✓")

---
## 10. Checkpoint Save / Load

In [ ]:
LOCAL_CKPT = "checkpoints"
mkpath(LOCAL_CKPT)

function save_checkpoint(path::String, model, opt_state;
                          step::Int=0, best_val_loss::Float64=Inf,
                          train_losses::Vector{Float64}=Float64[],
                          val_losses::Vector{Float64}=Float64[],
                          data_epoch::Int=0, data_pos_idx::Int=0)
    mkpath(dirname(path))
    model_cpu = cpu(model)
    opt_cpu = cpu(opt_state)
    jldopen(path, "w") do file
        file["model_state"] = Flux.state(model_cpu)
        file["opt_state"] = opt_cpu
        file["step"] = step
        file["best_val_loss"] = best_val_loss
        file["train_losses"] = train_losses
        file["val_losses"] = val_losses
        file["data_epoch"] = data_epoch
        file["data_pos_idx"] = data_pos_idx
        file["hyperparams"] = Dict(
            "vocab_size" => vocab_size,
            "n_embd" => n_embd,
            "block_size" => block_size,
            "n_layer" => n_layer,
            "n_head" => n_head,
            "n_kv_head" => n_kv_head,
            "dropout" => dropout,
            "use_bpe" => USE_BPE,
            "rope_base" => rope_base,
            "kd_alpha" => kd_alpha,
            "kd_temperature" => kd_temperature
        )
    end
    vl_str = best_val_loss == Inf ? "Inf" : @sprintf("%.4f", best_val_loss)
    println("Checkpoint saved: $path (step $step, best_val_loss=$vl_str)")
end

function save_and_sync(path, model, opt_state; kwargs...)
    save_checkpoint(path, model, opt_state; kwargs...)
    hf_sync(path)
end

function load_checkpoint(path::String, device_fn)
    println("Loading checkpoint from $path ...")
    data = JLD2.load(path)

    hp = data["hyperparams"]
    m = GPT(;
        vocab_size = hp["vocab_size"],
        n_embd     = hp["n_embd"],
        block_size = hp["block_size"],
        n_layer    = hp["n_layer"],
        n_head     = hp["n_head"],
        n_kv_head  = get(hp, "n_kv_head", hp["n_head"]),
        dropout    = get(hp, "dropout", 0.1)
    )

    # Load weights component-by-component (handles TiedDense weight sharing)
    ms = data["model_state"]
    Flux.loadmodel!(m.wte, ms[:wte])
    Flux.loadmodel!(m.drop, ms[:drop])
    Flux.loadmodel!(m.blocks, ms[:blocks])
    Flux.loadmodel!(m.ln_f, ms[:ln_f])
    # Skip lm_head — TiedDense shares wte.weight, already loaded

    m = m |> device_fn
    opt = data["opt_state"]

    println("  step=$(data["step"]), best_val=$(round(data["best_val_loss"], digits=4))")
    return (;
        model = m,
        opt_state = opt |> device_fn,
        step = data["step"],
        best_val_loss = data["best_val_loss"],
        train_losses = get(data, "train_losses", Float64[]),
        val_losses = get(data, "val_losses", Float64[]),
        data_epoch = get(data, "data_epoch", 0),
        data_pos_idx = get(data, "data_pos_idx", 0),
        hyperparams = hp
    )
end

println("Checkpoint save/load defined (JLD2 + HuggingFace sync)")

---
## 11. Training

Standard cross-entropy training. No checkpoint guard — this cell always trains from scratch.

In [ ]:
using Printf

# ── Generate text helper ──
function generate_text(model, max_tokens=200; temperature=0.8, prompt="")
    Flux.testmode!(model)
    try
        if !isempty(prompt)
            prompt_ids = encode(prompt)
            idx = reshape(prompt_ids, 1, :) |> device
        else
            idx = reshape([rand(1:vocab_size)], 1, 1) |> device
        end
        generated = Int[]
        for _ in 1:max_tokens
            idx_cond = idx[:, max(1, end-block_size+1):end]
            logits = model(idx_cond)
            logits_last = logits[:, end, 1]
            probs = softmax(logits_last ./ Float32(temperature))
            probs_cpu = Float64.(cpu(probs))
            r = rand()
            cum = 0.0
            next_id = length(probs_cpu)  # fallback to last token
            for (i, p) in enumerate(probs_cpu)
                cum += p
                if r <= cum
                    next_id = i
                    break
                end
            end
            push!(generated, next_id)
            next_token = reshape([next_id], 1, 1) |> device
            idx = hcat(idx, next_token)
        end
        return decode(generated)
    finally
        Flux.trainmode!(model)
    end
end

function estimate_loss(model, n_iters=eval_iters)
    Flux.testmode!(model)
    try
        losses = Dict{String, Float64}()
        for split in ["train", "val"]
            total = 0.0
            for _ in 1:n_iters
                x, y = get_batch(split)
                logits = model(x)
                y_flat = reshape(y, :)
                logits_flat = reshape(logits, vocab_size, :)
                onehot = Flux.onehotbatch(y_flat, 1:vocab_size) |> device
                loss = Flux.logitcrossentropy(logits_flat, onehot)
                total += loss
            end
            losses[split] = total / n_iters
        end
        losses["train_ppl"] = exp(losses["train"])
        losses["val_ppl"] = exp(losses["val"])
        return losses
    finally
        Flux.trainmode!(model)
    end
end

function get_lr(iter)
    if iter < warmup_iters
        return learning_rate * iter / warmup_iters
    end
    decay_ratio = (iter - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + cos(Float64(pi) * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)
end

opt_state = Flux.setup(
    OptimiserChain(ClipNorm(max_grad_norm), AdamW(learning_rate, (0.9f0, 0.999f0), 0.01f0)),
    model
)

best_val = Inf
train_loss_history = Float64[]
val_loss_history = Float64[]

if haskey(ENV, "WANDB_API_KEY") && !isempty(ENV["WANDB_API_KEY"])
    wandb_init(; project="JuliaDistillGPT", config=Dict{String,Any}(
        "compute_tier" => string(COMPUTE_TIER),
        "vram_gb" => VRAM_GB,
        "n_embd" => n_embd,
        "n_layer" => n_layer,
        "n_head" => n_head,
        "n_kv_head" => n_kv_head,
        "block_size" => block_size,
        "batch_size" => batch_size,
        "learning_rate" => learning_rate,
        "min_lr" => min_lr,
        "dropout" => dropout,
        "max_grad_norm" => max_grad_norm,
        "vocab_size" => vocab_size,
        "max_iters" => max_iters,
        "kd_alpha" => kd_alpha,
        "kd_temperature" => kd_temperature,
        "teacher_available" => teacher_logits_available,
        "architecture" => "DistillGPT-GQA-RoPE"
    ))
end

# ── Adaptive intervals for large datasets ──
SAVE_INTERVAL = 600
GC_INTERVAL = COMPUTE_TIER in (:t4, :medium) ? 50 : 100
global last_save_time = time()
global completed_iter = 0

mode_str = teacher_logits_available ? "KD (alpha=$kd_alpha, T=$kd_temperature)" : "Data distillation (standard CE)"
println("Training for $max_iters steps — mode: $mode_str")
println("DataLoader: epoch $(train_loader.epoch), $(length(train_loader.positions)) positions")
println("GC interval: every $GC_INTERVAL steps")
t_start = time()

try
    for iter in 1:max_iters
        global completed_iter = iter
        lr_t = get_lr(iter)
        Flux.adjust!(opt_state; eta=lr_t)

        x, y = get_batch("train")
        y_flat = reshape(y, :)
        onehot_y = Flux.onehotbatch(y_flat, 1:vocab_size) |> device

        # Teacher forward pass (outside gradient scope — no grads for teacher)
        if teacher_logits_available
            t_logits = teacher_model(x)
            t_logits_flat = reshape(t_logits, vocab_size, :)
        end

        loss, grads = Flux.withgradient(model) do m
            logits = m(x)
            logits_flat = reshape(logits, vocab_size, :)
            if teacher_logits_available
                kd_loss(logits_flat, t_logits_flat, onehot_y)
            else
                standard_loss(logits_flat, onehot_y)
            end
        end
        Flux.update!(opt_state, model, grads[1])
        push!(train_loss_history, Float64(loss))

        if iter % GC_INTERVAL == 0 && CUDA.functional()
            GC.gc(false)
        end

        if iter % eval_interval == 0 || iter == 1
            losses = estimate_loss(model)
            push!(val_loss_history, losses["val"])
            elapsed = round(time() - t_start, digits=1)
            wandb_log(; step=iter, train_loss=losses["train"], val_loss=losses["val"],
                       train_ppl=losses["train_ppl"], val_ppl=losses["val_ppl"], lr=lr_t,
                       data_epoch=train_loader.epoch)

            improved = ""
            if losses["val"] < best_val
                global best_val = losses["val"]
                save_and_sync("checkpoints/best_model.jld2", model, opt_state;
                    step=iter, best_val_loss=best_val,
                    train_losses=train_loss_history, val_losses=val_loss_history,
                    data_epoch=train_loader.epoch, data_pos_idx=train_loader.pos_idx)
                improved = " << best!"
            end

            @printf("step %5d | train %.4f (ppl %.1f) | val %.4f (ppl %.1f) | lr %.2e | ep %d | %.1fs%s\n",
                    iter, losses["train"], losses["train_ppl"],
                    losses["val"], losses["val_ppl"], lr_t, train_loader.epoch, elapsed, improved)
        end

        if iter % 1000 == 0
            save_and_sync("checkpoints/checkpoint_latest.jld2", model, opt_state;
                step=iter, best_val_loss=best_val,
                train_losses=train_loss_history, val_losses=val_loss_history,
                data_epoch=train_loader.epoch, data_pos_idx=train_loader.pos_idx)
            global last_save_time = time()
        end

        if time() - last_save_time > SAVE_INTERVAL
            save_and_sync("checkpoints/checkpoint_latest.jld2", model, opt_state;
                step=iter, best_val_loss=best_val,
                train_losses=train_loss_history, val_losses=val_loss_history,
                data_epoch=train_loader.epoch, data_pos_idx=train_loader.pos_idx)
            global last_save_time = time()
        end
    end
catch e
    if e isa InterruptException
        println("\nInterrupted at step $completed_iter!")
    else
        println("\nError at step $completed_iter: $e")
    end
    save_and_sync("checkpoints/checkpoint_interrupted.jld2", model, opt_state;
        step=completed_iter, best_val_loss=best_val,
        train_losses=train_loss_history, val_losses=val_loss_history,
        data_epoch=train_loader.epoch, data_pos_idx=train_loader.pos_idx)
    e isa InterruptException || rethrow(e)
end

elapsed = round(time() - t_start, digits=1)
println("\nTraining complete in $(elapsed)s. Best val: $(round(best_val, digits=4)) (ppl $(round(exp(best_val), digits=1)))")
println("Final data epoch: $(train_loader.epoch)")
wandb_finish()

save_and_sync("checkpoints/final_model.jld2", model, opt_state;
    step=max_iters, best_val_loss=best_val,
    train_losses=train_loss_history, val_losses=val_loss_history,
    data_epoch=train_loader.epoch, data_pos_idx=train_loader.pos_idx)


---
## 12. Generate Text

In [ ]:
sample = generate_text(model, 200; temperature=0.8, prompt="the good is")
println("Sample:\n", sample)

---
## 13. Push Model to HuggingFace

In [ ]:
if @isdefined(HF_REPO_ID) && !isempty(HF_REPO_ID)
    hf_create_repo(HF_REPO_ID)

    if isfile("checkpoints/best_model.jld2")
        hf_push_checkpoint(HF_REPO_ID; checkpoint_path="checkpoints/best_model.jld2")
    else
        println("No best_model.jld2 found — train first!")
    end

    if isfile("checkpoints/final_model.jld2")
        hf_push(HF_REPO_ID, "checkpoints/final_model.jld2")
    end

    println("\nDone! View your model at: https://huggingface.co/$HF_REPO_ID")
else
    println("Set HF_REPO_ID in the credentials cell")
end

---
## 14. Pull Checkpoint from HuggingFace

In [ ]:
if @isdefined(HF_REPO_ID) && !isempty(HF_REPO_ID)
    mkpath("checkpoints")
    hf_pull(HF_REPO_ID, "best_model.jld2"; local_dir="checkpoints")
    println("\nReady to resume from checkpoints/best_model.jld2")
    println("Run the 'Resume Training' cell below.")
else
    println("Set HF_REPO_ID in the credentials cell")
end

---
## 15. Resume Training from Checkpoint

**Set `RESUME_TRAINING = true` to enable.** Skipped on "Run All" by default.

In [ ]:
# ── Set to true to resume training ──
RESUME_TRAINING = false

if !RESUME_TRAINING
    println("Resume training skipped (RESUME_TRAINING = false)")
else

RESUME_FROM = "checkpoints/best_model.jld2"
EXTRA_ITERS = 2000

if !isfile(RESUME_FROM)
    if @isdefined(HF_REPO_ID) && !isempty(HF_REPO_ID)
        println("Checkpoint not found locally, pulling from HuggingFace...")
        hf_pull(HF_REPO_ID, basename(RESUME_FROM); local_dir="checkpoints")
    end
    isfile(RESUME_FROM) || error("Checkpoint not found: $RESUME_FROM")
end

ckpt = load_checkpoint(RESUME_FROM, device)

# ── Verify vocab compatibility ──
ckpt_vocab = Int(ckpt.hyperparams["vocab_size"])
if ckpt_vocab != vocab_size
    error("Vocab mismatch! Checkpoint has vocab_size=$ckpt_vocab but tokenizer has vocab_size=$vocab_size. " *
          "Delete old checkpoint or retrain with matching tokenizer.")
end

model = ckpt.model
opt_state = ckpt.opt_state
start_iter = ckpt.step + 1
best_val = ckpt.best_val_loss
train_loss_history = copy(ckpt.train_losses)
val_loss_history = copy(ckpt.val_losses)
end_iter = ckpt.step + EXTRA_ITERS

# ── Recreate dataloader ──
global train_loader = ShuffledDataLoader(train_data, block_size, batch_size)

# ── Restore dataloader state ──
ckpt_hp = ckpt.hyperparams
if ckpt.data_epoch > 0
    train_loader.epoch = ckpt.data_epoch
    rng = MersenneTwister(1337 + train_loader.epoch)
    shuffle!(rng, train_loader.positions)
    if ckpt.data_pos_idx > 0 && ckpt.data_pos_idx <= length(train_loader.positions)
        train_loader.pos_idx = ckpt.data_pos_idx
    end
end

# ── Fresh cosine LR schedule (30% of original peak) ──
resume_warmup_steps = max(50, round(Int, 0.02 * EXTRA_ITERS))
resume_peak_lr = learning_rate * 0.3

function get_lr_resume(iter)
    local_iter = iter - start_iter + 1
    if local_iter <= resume_warmup_steps
        return resume_peak_lr * local_iter / resume_warmup_steps
    end
    decay_ratio = (local_iter - resume_warmup_steps) / max(1, EXTRA_ITERS - resume_warmup_steps)
    coeff = 0.5 * (1.0 + cos(Float64(pi) * min(decay_ratio, 1.0)))
    return min_lr + coeff * (resume_peak_lr - min_lr)
end

println("\nResuming from step $(ckpt.step) -> training to step $end_iter")
println("Best val loss so far: $(round(best_val, digits=4))")
t_start = time()
SAVE_INTERVAL = 600
last_save_time = time()

try
    for iter in start_iter:end_iter
        global completed_iter = iter
        lr_t = get_lr_resume(iter)
        Flux.adjust!(opt_state; eta=lr_t)

        x, y = get_batch("train")
        y_flat = reshape(y, :)
        onehot_y = Flux.onehotbatch(y_flat, 1:vocab_size) |> device
        loss, grads = Flux.withgradient(model) do m
            logits = m(x)
            logits_flat = reshape(logits, vocab_size, :)
            standard_loss(logits_flat, onehot_y)
        end
        Flux.update!(opt_state, model, grads[1])
        push!(train_loss_history, Float64(loss))

        if iter % 100 == 0 && CUDA.functional()
            GC.gc(false)
        end

        if iter % eval_interval == 0 || iter == start_iter
            losses = estimate_loss(model)
            push!(val_loss_history, losses["val"])
            elapsed = round(time() - t_start, digits=1)

            improved = ""
            if losses["val"] < best_val
                best_val = losses["val"]
                save_and_sync("checkpoints/best_model.jld2", model, opt_state;
                    step=iter, best_val_loss=best_val,
                    train_losses=train_loss_history, val_losses=val_loss_history,
                    data_epoch=train_loader.epoch, data_pos_idx=train_loader.pos_idx)
                improved = " << best!"
            end

            @printf("step %5d / %5d | train %.4f (ppl %.1f) | val %.4f (ppl %.1f) | lr %.2e | ep %d | %.1fs%s\n",
                    iter, end_iter, losses["train"], losses["train_ppl"],
                    losses["val"], losses["val_ppl"], lr_t, train_loader.epoch, elapsed, improved)
        end

        if time() - last_save_time > SAVE_INTERVAL
            save_and_sync("checkpoints/checkpoint_latest.jld2", model, opt_state;
                step=iter, best_val_loss=best_val,
                train_losses=train_loss_history, val_losses=val_loss_history,
                data_epoch=train_loader.epoch, data_pos_idx=train_loader.pos_idx)
            last_save_time = time()
        end
    end
catch e
    if e isa InterruptException
        println("\nInterrupted at step $completed_iter!")
    else
        println("\nError at step $completed_iter: $e")
    end
    save_and_sync("checkpoints/checkpoint_interrupted.jld2", model, opt_state;
        step=completed_iter, best_val_loss=best_val,
        train_losses=train_loss_history, val_losses=val_loss_history,
        data_epoch=train_loader.epoch, data_pos_idx=train_loader.pos_idx)
    e isa InterruptException || rethrow(e)
end

elapsed = round(time() - t_start, digits=1)
println("\nResume complete in $(elapsed)s. Best val: $(round(best_val, digits=4))")
wandb_finish()

end  # RESUME_TRAINING guard